# مرحله ۲: ساخت مدل‌های پایه (Baseline Models)

### معرفی پروژه و هدف Baseline


# Baseline Models – Digikala Review Analysis

## Objectives
Build two baseline text classification models using TF-IDF and Logistic Regression.

### Task 1: Recommendation Status
Predict one of:
- recommended
- not_recommended
- no_idea

### Task 2: Rating Band
Predict one of:
- low
- medium
- high

## Evaluation
- Accuracy
- Macro F1
- Precision
- Recall
- Confusion Matrix

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# پیدا کردن پوشه اصلی پروژه
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "comments_ready.parquet"

print("Project root:", PROJECT_ROOT)
print("Data file exists:", DATA_PATH.exists())
print("Data path:", DATA_PATH)

Project root: c:\Users\Lenovo\Desktop\project3\project-3
Data file exists: True
Data path: c:\Users\Lenovo\Desktop\project3\project-3\data\processed\comments_ready.parquet


### بررسی ساختار فایل آماده‌شده

این سلول نام ستون‌ها و نوع داده‌های فایل را نشان می‌دهد. می‌خواهیم مطمئن شویم ستون‌های موردنیاز را داریم:
- split
- text_v1
- text_v2
- recommendation_status
- rate

چرا فعلاً آموزش را شروع نمی‌کنیم؟ چون قبل از انتقال به Colab باید دقیقاً بدانیم چه ستون‌هایی داریم و داده‌ها چگونه تقسیم شده‌اند.

In [3]:
con = duckdb.connect()

schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DATA_PATH.as_posix()}')
""").fetchdf()

display(schema)

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,body,VARCHAR,YES,None,None,None
3,created_at,VARCHAR,YES,None,None,None
4,rate,DOUBLE,YES,None,None,None
5,recommendation_status,VARCHAR,YES,None,None,None
6,is_buyer,BOOLEAN,YES,None,None,None
7,product_id,BIGINT,YES,None,None,None
8,advantages,VARCHAR,YES,None,None,None
9,disadvantages,VARCHAR,YES,None,None,None


خروجی کاملاً خوبه. هر سه ستون اصلی موردنیاز ما وجود دارند:

- split: برای تشخیص Train، Validation و Test
- text_v1 و text_v2: دو نسخه متن برای مقایسه نرمال‌سازی
- recommendation_status: برچسب مسئله اول
- rate: مبنای ساخت برچسب مسئله دوم

یک نکته مهم: چون دو نسخه نرمال‌سازی داریم، باید هر دو را جداگانه آزمایش کنیم تا بفهمیم کدام‌یک عملکرد بهتری دارد.

## گام دوم: بررسی تعداد داده‌های قابل استفاده برای مدل‌ها
قبل از ساخت TF-IDF باید تعداد نمونه‌های معتبر هر مجموعه را بدانیم.

این کد چه چیزی را بررسی می‌کند؟

- total_rows	کل نظرات هر مجموعه
- valid_text_rows	نظراتی که متن معتبر دارند
- recommendation_usable	نظراتی که هم متن معتبر و هم برچسب پیشنهاد دارند
- rating_usable	نظراتی که هم متن معتبر و هم امتیاز دارند

دقت کن که تعداد داده‌های دو مسئله برابر نیست؛ چون بعضی نظرات برچسب recommendation_status ندارند، اما همچنان برای پیش‌بینی امتیاز قابل استفاده‌اند.

In [4]:
counts = con.execute(f"""
    SELECT
        split,

        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE text_v1 IS NOT NULL
              AND trim(text_v1) != ''
              AND lower(trim(text_v1)) NOT IN ('nan', 'null')
        ) AS valid_text_rows,

        COUNT(*) FILTER (
            WHERE text_v1 IS NOT NULL
              AND trim(text_v1) != ''
              AND lower(trim(text_v1)) NOT IN ('nan', 'null')
              AND recommendation_status IS NOT NULL
              AND lower(trim(recommendation_status))
                  IN ('recommended', 'not_recommended', 'no_idea')
        ) AS recommendation_usable,

        COUNT(*) FILTER (
            WHERE text_v1 IS NOT NULL
              AND trim(text_v1) != ''
              AND lower(trim(text_v1)) NOT IN ('nan', 'null')
              AND rate IS NOT NULL
        ) AS rating_usable

    FROM read_parquet('{DATA_PATH.as_posix()}')
    GROUP BY split
    ORDER BY
        CASE split
            WHEN 'train' THEN 1
            WHEN 'validation' THEN 2
            WHEN 'test' THEN 3
        END
""").fetchdf()

display(counts)

,split,total_rows,valid_text_rows,recommendation_usable,rating_usable
0,train,4308541,4308085,3681815,4308085
1,validation,922757,922670,789277,922670
2,test,923673,923579,789129,923579


از نظر تعداد، نتایج با مجموع داده‌های پروژه سازگارند.

اما چرا تعداد ردیف‌های Train و Validation و Test با خروجی اولیه تقسیم داده‌ها فرق دارد؟

در تقسیم اولیه داشتیم:

- Train: 4,312,367
- Validation: 912,255
- Test: 930,349

در خروجی جدید داریم:

- Train: 4,308,541
- Validation: 922,757
- Test: 923,673

مجموع هر دو برابر است، ولی توزیع بین مجموعه‌ها تغییر کرده است. احتمالاً علت، تغییر فایل یا منطق تقسیم است. فعلاً این موضوع را نادیده نمی‌گیریم؛ باید مشخص کنیم فایل فعلی همان تقسیم محصول‌محور اولیه است یا نه.

## ۲. یک کنترل مهم قبل از ادامه
بررسی مجدد تعداد محصولات و نشت داده   

In [5]:
split_check = con.execute(f"""
    SELECT
        split,
        COUNT(*) AS comments,
        COUNT(DISTINCT product_id) AS products
    FROM read_parquet('{DATA_PATH.as_posix()}')
    GROUP BY split
    ORDER BY
        CASE split
            WHEN 'train' THEN 1
            WHEN 'validation' THEN 2
            WHEN 'test' THEN 3
        END
""").fetchdf()

display(split_check)


leakage = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            product_id
        FROM read_parquet('{DATA_PATH.as_posix()}')
        GROUP BY product_id
        HAVING COUNT(DISTINCT split) > 1
    )
""").fetchone()[0]

print("Leaked products:", leakage)

,split,comments,products
0,train,4308541,232119
1,validation,922757,49739
2,test,923673,49741


Leaked products: 0


> نتیجه نشان می‌دهد که تقسیم فعلی همچنان محصول‌محور است و هیچ محصولی بین سه مجموعه مشترک نیست:

> تعداد محصولات هم دقیقاً با تقسیم اولیه مطابقت دارد:

> پس اختلافی که در تعداد کامنت‌های سه مجموعه دیدیم، ناشی از تغییر تعداد محصولات نیست؛ تعداد کامنت‌های تخصیص‌یافته به هر مجموعه تغییر کرده، اما تخصیص محصولات و جلوگیری از نشت داده درست است.

## گام بعدی: ساخت برچسب Rating Band
حالا باید امتیازهای عددی را به سه کلاس تبدیل کنیم.

قانونی که از ابتدای پروژه تعیین کردیم:
- 0 <= rate < 2.5	low
- 2.5 <= rate < 4	medium
- 4 <= rate <= 5	high

می‌خواهیم مطمئن شویم هر سه کلاس در Train، Validation و Test به اندازه کافی نمونه دارند. همچنین مشخص کنیم آیا عدم تعادل کلاس‌ها باید در تنظیمات Logistic Regression در نظر گرفته شود یا نه.

In [6]:
rating_distribution = con.execute(f"""
    SELECT
        split,
        CASE
            WHEN rate < 2.5 THEN 'low'
            WHEN rate < 4.0 THEN 'medium'
            ELSE 'high'
        END AS rating_band,
        COUNT(*) AS count
    FROM read_parquet('{DATA_PATH.as_posix()}')
    WHERE text_v1 IS NOT NULL
      AND trim(text_v1) != ''
      AND lower(trim(text_v1)) NOT IN ('nan', 'null')
    GROUP BY split, rating_band
    ORDER BY
        CASE split
            WHEN 'train' THEN 1
            WHEN 'validation' THEN 2
            WHEN 'test' THEN 3
        END,
        CASE rating_band
            WHEN 'low' THEN 1
            WHEN 'medium' THEN 2
            WHEN 'high' THEN 3
        END
""").fetchdf()

display(rating_distribution)

,split,rating_band,count
0,train,low,767438
1,train,medium,740921
2,train,high,2799726
3,validation,low,161881
4,validation,medium,158665
5,validation,high,602124
6,test,low,162740
7,test,medium,159108
8,test,high,601731


In [ ]:
# برای اینکه بهتر ببینی، درصد هر کلاس را محاسبه کنیم:

rating_distribution["percentage"] = (
    rating_distribution["count"]
    / rating_distribution.groupby("split")["count"].transform("sum")
    * 100
)

display(rating_distribution)

,split,rating_band,count,percentage
0,train,low,767438,17.813901
1,train,medium,740921,17.198384
2,train,high,2799726,64.987715
3,validation,low,161881,17.544843
4,validation,medium,158665,17.196289
5,validation,high,602124,65.258868
6,test,low,162740,17.620583
7,test,medium,159108,17.227330
8,test,high,601731,65.152088


نتیجه: توزیع کلاس‌ها بین سه مجموعه متعادل و مشابه است؛ اما خود مسئله نامتوازن است، چون کلاس high تقریباً ۶۵ درصد داده‌ها را تشکیل می‌دهد.

بنابراین هنگام ارزیابی فقط به Accuracy تکیه نمی‌کنیم و Macro F1 را معیار مهم‌تری در نظر می‌گیریم.

## ۲. مرحله بعد: بررسی توزیع Recommendation
حالا باید همین بررسی را برای مسئله اول انجام دهیم.

چرا این بررسی مهم است؟

در تحلیل اولیه دیدیم که کلاس recommended حدود ۷۹ درصد برچسب‌های معتبر را تشکیل می‌دهد.

می‌خواهیم مطمئن شویم این نسبت در Train، Validation و Test تقریباً ثابت مانده است.

In [8]:
recommendation_distribution = con.execute(f"""
    SELECT
        split,
        recommendation_status,
        COUNT(*) AS count
    FROM read_parquet('{DATA_PATH.as_posix()}')
    WHERE text_v1 IS NOT NULL
      AND trim(text_v1) != ''
      AND lower(trim(text_v1)) NOT IN ('nan', 'null')
      AND recommendation_status IN (
          'recommended',
          'not_recommended',
          'no_idea'
      )
    GROUP BY split, recommendation_status
    ORDER BY
        CASE split
            WHEN 'train' THEN 1
            WHEN 'validation' THEN 2
            WHEN 'test' THEN 3
        END,
        CASE recommendation_status
            WHEN 'not_recommended' THEN 1
            WHEN 'no_idea' THEN 2
            WHEN 'recommended' THEN 3
        END
""").fetchdf()

recommendation_distribution["percentage"] = (
    recommendation_distribution["count"]
    / recommendation_distribution.groupby("split")["count"].transform("sum")
    * 100
)

display(recommendation_distribution)

,split,recommendation_status,count,percentage
0,train,not_recommended,353170,9.592280
1,train,no_idea,415233,11.277943
2,train,recommended,2913412,79.129777
3,validation,not_recommended,75417,9.555201
4,validation,no_idea,89306,11.314912
5,validation,recommended,624554,79.129887
6,test,not_recommended,76209,9.657356
7,test,no_idea,89406,11.329707
8,test,recommended,623514,79.012937


: مسئله Recommendation نامتوازن است. تقریباً ۷۹٪ داده‌ها متعلق به کلاس recommended هستند. اگر مدلی همه نظرات را recommended پیش‌بینی کند، Accuracy ظاهراً خوبی می‌گیرد، ولی در تشخیص دو کلاس دیگر ضعیف خواهد بود.

به همین دلیل در ارزیابی هر دو مسئله، Macro F1 را معیار اصلی مقایسه قرار می‌دهیم و در کنار آن Precision، Recall و Confusion Matrix را هم بررسی می‌کنیم.

---
```text
چهار آزمایش خواهیم داشت:
آزمایش	متن ورودی	هدف
1	text_v1	Recommendation
2	text_v2	Recommendation
3	text_v1	Rating Band
4	text_v2	Rating Band
```

## ۳. قبل از انتقال به Google Colab: بررسی حجم فایل
چون قرار است آموزش روی Colab انجام شود، باید بدانیم فایل Parquet چقدر فضا اشغال می‌کند. لازم نیست CSVهای اصلی ۱.۲ گیگابایتی و ۲۴۷ مگابایتی را منتقل کنیم؛ فایل آماده‌شده ما comments_ready.parquet است.

In [9]:
file_size_bytes = DATA_PATH.stat().st_size

print(f"File size: {file_size_bytes / (1024**3):.2f} GB")
print(f"File size: {file_size_bytes / (1024**2):.2f} MB")

File size: 0.57 GB
File size: 578.75 MB


مسیر ادامه پروژه
1. تأیید توزیع کلاس‌ها انجام شد
2. بررسی حجم فایل آماده‌شده گام فعلی
3. آماده‌سازی Google Colab و اتصال Google Drive
4. انتقال Parquet و بارگذاری داده‌ها
5. اجرای چهار آزمایش Baseline
6. مقایسه نتایج و انتخاب Baseline مناس

# Checkpoint-based Workflow for saving

## ۳. یک تغییر مهم: فایل ۵۷۹ مگابایتی را تقسیم می‌کنیم
پیشنهاد من این است که فایل comments_ready.parquet را به سه فایل مجزا تبدیل کنیم:

- train.parquet
- validation.parquet
- test.parquet

مزیتش این است که اگر آپلود یکی از فایل‌ها قطع شود، لازم نیست هر سه را دوباره منتقل کنیم.

این فایل‌ها فقط نسخه‌های تقسیم‌شده برای انتقال هستند؛ فایل اصلی را دست‌نخورده نگه می‌داریم.

## ۴. گام فعلی: تقسیم فایل در کامپیوتر خودت

In [10]:
# پوشه مخصوص فایل‌های قابل انتقال به Colab
COLAB_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "colab_upload"

COLAB_DATA_DIR.mkdir(parents=True, exist_ok=True)

splits = ["train", "validation", "test"]

for split_name in splits:

    output_path = COLAB_DATA_DIR / f"{split_name}.parquet"

    query = f"""
        COPY (
            SELECT *
            FROM read_parquet('{DATA_PATH.as_posix()}')
            WHERE split = '{split_name}'
        )
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD);
    """

    con.execute(query)

    print(
        f"{split_name}: "
        f"{output_path.stat().st_size / (1024**2):.2f} MB"
    )

train: 404.18 MB
validation: 86.95 MB
test: 86.40 MB


## ۵. کنترل صحت فایل‌های جداشده

In [11]:
for split_name in splits:

    file_path = COLAB_DATA_DIR / f"{split_name}.parquet"

    result = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT product_id) AS products
        FROM read_parquet('{file_path.as_posix()}')
    """).fetchdf()

    print(f"\n{split_name.upper()}")
    display(result)


TRAIN


,rows,products
0,4308541,232119



VALIDATION


,rows,products
0,922757,49739



TEST


,rows,products
0,923673,49741
